# Generacion de Video desde Prompt (Text-to-Video)

Este notebook esta enfocado solo en generar video desde texto con Diffusers.

Flujo:
1. Instalar dependencias.
2. Cargar pipeline Text-to-Video.
3. Generar frames desde prompt.
4. Exportar MP4 y revisar tamano/duracion.

## Chuleta rapida: que modelo usar segun objetivo

| Objetivo | Recomendacion | Modelos sugeridos | Comentario |
|---|---|---|---|
| Maxima calidad (imagen) | Prioriza detalle y fidelidad | black-forest-labs/FLUX.1-dev, stabilityai/stable-diffusion-xl-base-1.0 | Mas lento y mas VRAM |
| Maxima velocidad (imagen) | Menos pasos, modelo turbo | Tongyi-MAI/Z-Image-Turbo, stabilityai/sdxl-turbo | Ideal para iterar prompts |
| Menor VRAM (imagen) | Resolucion media/baja y pasos moderados | dreamlike-art/dreamlike-diffusion-1.0, CompVis/stable-diffusion-v1-4 | Mas amigable en GPUs ajustadas |
| Video consistente en estilo | Flujo imagen -> video | stabilityai/stable-video-diffusion-img2vid-xt | Mantiene mejor composicion/estetica |
| Video directo desde prompt | Text-to-video | damo-vilab/text-to-video-ms-1.7b | Mas simple, pero mas pesado y variable |

Notas practicas:
1. Si quieres mejor acabado final: prompt -> imagen (FLUX/SDXL) -> video (SVD img2vid).
2. Si quieres velocidad para probar ideas: Z-Image-Turbo y clips cortos.
3. Para videos largos: genera segmentos cortos y luego concatena.

#### Ejemplos

stabilityai/stable-video-diffusion-img2vid
ali-vilab/modelscope-damo-text-to-video-synthesis
Alissonerdx/LTX-Best-Face-ID # generador de rostros
realrebelai/LingBot-30B-3B_GGUF_ComfyUI # generador de animales
SOLRICKS/ltx-2.3-product-ad-style # para publicidad de marcas
Wan-AI/Wan2.2-T2V-A14B # modelo chino
jdopensource/JoyAI-Echo
Qualcomm-AI-Research/mobilewan






In [ ]:
%pip install -q diffusers transformers accelerate imageio imageio-ffmpeg safetensors

In [ ]:
import os
import torch
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video
from IPython.display import Video, display

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Este pipeline Text-to-Video requiere GPU CUDA para funcionar de forma razonable.')

## 1) Cargar modelo Text-to-Video

Modelo base sugerido: `damo-vilab/text-to-video-ms-1.7b`

Nota: es pesado en VRAM. Se habilita offload para hacerlo mas estable en GPUs ajustadas.

In [ ]:
model_id = 'damo-vilab/text-to-video-ms-1.7b'

pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant='fp16'
)

pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.enable_attention_slicing()

print('Pipeline cargado:', model_id)

## 2) Definir prompt y parametros

Puedes cambiar el prompt para mantener el estilo que quieras en el video.

In [ ]:
prompt = (
    'A futuristic city overtaken by nature, giant broken robots, a lone wanderer with cape, '
    'cinematic lighting, atmospheric fog, hyperrealistic details, dramatic clouds'
)

negative_prompt = 'blurry, low quality, distorted, artifacts, watermark, text'

num_frames = 16
num_inference_steps = 25
guidance_scale = 9.0
fps = 8
seed = 42

print('[PROMPT]:', prompt)
print('Frames:', num_frames, '| Steps:', num_inference_steps, '| FPS:', fps)

In [ ]:
# Guia visual: duracion (segundos) = num_frames / fps
# Ejemplos utiles:
# - 24 frames / 8 fps = 3.0 s
# - 12 frames / 6 fps = 2.0 s
# - 80 frames / 8 fps = 10.0 s
# - 240 frames / 8 fps = 30.0 s

profiles = {
    "quality": {
        "num_frames": 24,
        "num_inference_steps": 40,
        "guidance_scale": 10.0,
        "fps": 8,
    },
    "fast": {
        "num_frames": 12,
        "num_inference_steps": 20,
        "guidance_scale": 8.0,
        "fps": 6,
    },
}

selected_profile = "fast"  # Cambia a "quality" o "fast"
if selected_profile not in profiles:
    raise ValueError(f"Perfil no valido: {selected_profile}. Usa 'quality' o 'fast'.")

cfg = profiles[selected_profile]
print("Perfil seleccionado:", selected_profile)
print("Configuracion:", cfg)
print(f"Duracion aproximada: {cfg['num_frames'] / cfg['fps']:.2f} s")

## 3) Elegir perfil: Quality o Fast

Selecciona el perfil en la siguiente celda cambiando solo la variable selected_profile.

- quality: mejor detalle y duracion un poco mayor (mas tiempo de generacion).
- fast: menor costo y generacion mas rapida.

In [ ]:
generator = torch.Generator(device='cuda').manual_seed(seed)

video_frames = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_frames=cfg['num_frames'],
    num_inference_steps=cfg['num_inference_steps'],
    guidance_scale=cfg['guidance_scale'],
    generator=generator
).frames[0]

output_video_path = f"video_desde_prompt_{selected_profile}.mp4"
export_to_video(video_frames, output_video_path=output_video_path, fps=cfg['fps'])

file_size_mb = os.path.getsize(output_video_path) / (1024 * 1024)
duration_seconds = cfg['num_frames'] / cfg['fps']

print('Video generado:', output_video_path)
print('Perfil:', selected_profile)
print(f'Duracion aproximada: {duration_seconds:.2f} s')
print(f'Tamano: {file_size_mb:.2f} MB')

In [ ]:
display(Video(output_video_path, embed=True))